# Phase 2: MAPPO robustness, transfer, and reports
Each experiment section can run after initialization in a fresh runtime. A fresh, post-repair Phase-1 baseline run and nominal IPPO run are mandatory: the gate below stops execution unless every safeguard passes. Never interpret Pymunk transfer as physical deployment.

## 1. Compact initialization

In [ ]:
from pathlib import Path
import json, shutil, sys
REPO_URL = "https://github.com/djdhillxn/football"
REPO_DIR = Path("/content/robot-soccer-transfer")
DRIVE_MOUNT = Path("/content/drive")
DRIVE_PROJECT = DRIVE_MOUNT / "MyDrive" / "RobotSoccerTransfer"
!python --version
!nvidia-smi || true
from google.colab import drive
drive.mount(str(DRIVE_MOUNT))
if REPO_DIR.exists():
    dirty = !git -C {REPO_DIR} status --porcelain
    if dirty:
        raise RuntimeError("Repository has local changes; refusing to overwrite them.")
    !git -C {REPO_DIR} fetch origin --quiet
    !git -C {REPO_DIR} pull --ff-only
else:
    if REPO_URL.startswith("REPLACE_"):
        raise RuntimeError("Set REPO_URL before cloning.")
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git rev-parse HEAD
# Preserve Colab's CUDA-enabled Torch, NumPy, and pandas builds.
!python -m pip install -q gymnasium==1.2.1 pettingzoo==1.26.1 pymunk==7.3.0 PyYAML==6.0.3 tensorboard==2.20.0 imageio==2.37.2 imageio-ffmpeg==0.6.0 pytest==8.4.2 ruff==0.14.5
if int(get_ipython().user_ns.get("_exit_code", 0)) != 0:
    raise RuntimeError("dependency installation failed")
!python -m pip install -e . --no-deps -q
if int(get_ipython().user_ns.get("_exit_code", 0)) != 0:
    raise RuntimeError("editable package installation failed")
sys.path.insert(0, str(REPO_DIR))
import torch, robosoccer
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), "project", robosoccer.__version__)

In [ ]:
def require_shell_success(label):
    status = int(get_ipython().user_ns.get("_exit_code", 0))
    if status != 0:
        raise RuntimeError(f"{label} failed with shell status {status}")

def latest_run(name):
    pointer = REPO_DIR / "runs" / f"latest_{name}.txt"
    if not pointer.is_file():
        raise FileNotFoundError(pointer)
    return Path(pointer.read_text().strip())

def restore_run(name):
    source = DRIVE_PROJECT / "runs" / name
    destination = REPO_DIR / "runs" / name
    shutil.copytree(source, destination, dirs_exist_ok=True)
    metadata = json.loads((destination / "run_metadata.json").read_text())
    pointer = REPO_DIR / "runs" / f"latest_{metadata['experiment_name']}.txt"
    pointer.parent.mkdir(parents=True, exist_ok=True)
    pointer.write_text(str(destination.resolve()) + "\n")
    return destination

def copy_completed_run(run_dir):
    run_dir = Path(run_dir)
    metadata = json.loads((run_dir / "run_metadata.json").read_text())
    if metadata.get("status") != "complete":
        raise RuntimeError(f"Incomplete run: {run_dir}")
    destination = DRIVE_PROJECT / "runs" / run_dir.name
    shutil.copytree(run_dir, destination, dirs_exist_ok=True)
    return destination

def display_json(path):
    data = json.loads(Path(path).read_text())
    print(json.dumps(data, indent=2)[:5000])
    return data

from robosoccer.config import load_config
from robosoccer.evaluation import phase1_readiness_audit

def require_phase1_go():
    baseline_run = latest_run("base")
    ippo_run = latest_run("ippo_nominal")
    baseline_summary = json.loads((baseline_run / "eval" / "baseline_summary.json").read_text())
    abstract_summary = json.loads((ippo_run / "eval" / "abstract_standard" / "summary.json").read_text())
    transfer_summary = json.loads((ippo_run / "eval" / "pymunk_transfer" / "summary.json").read_text())
    audit = phase1_readiness_audit(
        load_config(REPO_DIR / "configs" / "base.yaml"),
        baseline_summary,
        abstract_summary,
        transfer_summary,
    )
    print(json.dumps(audit, indent=2))
    if not audit["phase2_ready"]:
        raise RuntimeError("PHASE 2 NO-GO: rerun or repair Phase 1 before training MAPPO.")
    print("PHASE 2 GO: all Phase-1 safeguards passed.")
    return audit

def sync_all():
    shutil.copytree(REPO_DIR / "reports", DRIVE_PROJECT / "reports", dirs_exist_ok=True)
    comparisons = REPO_DIR / "runs" / "comparisons"
    if comparisons.exists():
        shutil.copytree(comparisons, DRIVE_PROJECT / "comparisons", dirs_exist_ok=True)

## 2. Restore the required Phase-1 artifacts and enforce the gate

In [ ]:
# Add the fresh post-repair baseline and nominal-IPPO Drive directory names here.
RUNS_TO_RESTORE = []
for run_name in RUNS_TO_RESTORE:
    print(restore_run(run_name))
phase1_audit = require_phase1_go()

## 3. Train nominal MAPPO
This section is reachable only after the Phase-1 audit reports `phase2_ready: true`.

In [ ]:
!python -m scripts.train --config configs/mappo_nominal.yaml
require_shell_success("nominal MAPPO training")

## 4. Train uniformly randomized MAPPO

In [ ]:
!python -m scripts.train --config configs/mappo_uniform_dr.yaml
require_shell_success("uniform-randomization MAPPO training")

## 5. Train failure-directed MAPPO

In [ ]:
!python -m scripts.train --config configs/mappo_failure_dr.yaml
require_shell_success("failure-directed MAPPO training")

## 6. Optional focused ablations

In [ ]:
# Run only after the primary methods and only if the experiment budget allows.
!python -m scripts.train --config configs/mappo_failure_dr.yaml --run-name mappo_failure_no_action_delay 'randomization.disabled_families=[action_delay]'
require_shell_success("action-delay ablation")
!python -m scripts.train --config configs/mappo_failure_dr.yaml --run-name mappo_failure_no_communication 'randomization.disabled_families=[communication]'
require_shell_success("communication ablation")
# Optional: replace the family below with observation_noise.
#!python -m scripts.train --config configs/mappo_failure_dr.yaml --run-name mappo_failure_no_observation_noise 'randomization.disabled_families=[observation_noise]'

## 7. Evaluate learned policies in the abstract simulator

In [ ]:
for experiment in ["mappo_nominal", "mappo_uniform_dr", "mappo_failure_dr"]:
    run_dir = latest_run(experiment)
    !python -m scripts.evaluate --run-dir {run_dir} --checkpoint best --simulator abstract --suite standard
    require_shell_success(f"{experiment} abstract evaluation")

## 8. Evaluate the same frozen actors in Pymunk

In [ ]:
for experiment in ["mappo_nominal", "mappo_uniform_dr", "mappo_failure_dr"]:
    run_dir = latest_run(experiment)
    !python -m scripts.evaluate --run-dir {run_dir} --checkpoint best --simulator pymunk --suite transfer
    require_shell_success(f"{experiment} Pymunk transfer evaluation")

## 9. Run profile evaluations

In [ ]:
for experiment in ["mappo_nominal", "mappo_uniform_dr", "mappo_failure_dr"]:
    run_dir = latest_run(experiment)
    !python -m scripts.evaluate --run-dir {run_dir} --checkpoint best --simulator pymunk --suite profiles
    require_shell_success(f"{experiment} profile evaluation")

## 10. Run robustness grids

In [ ]:
for experiment in ["mappo_nominal", "mappo_uniform_dr", "mappo_failure_dr"]:
    run_dir = latest_run(experiment)
    !python -m scripts.evaluate --run-dir {run_dir} --checkpoint best --simulator pymunk --suite robustness
    require_shell_success(f"{experiment} robustness grid")

## 11. Record nominal and transfer videos

In [ ]:
failure_run = latest_run("mappo_failure_dr")
!python -m scripts.record_video --run-dir {failure_run} --checkpoint best --simulator abstract --episodes 3
require_shell_success("failure-directed abstract videos")
!python -m scripts.record_video --run-dir {failure_run} --checkpoint best --simulator pymunk --episodes 3
require_shell_success("failure-directed transfer videos")

## 12. Compare all completed runs

In [ ]:
!python -m scripts.compare_runs --phase final --export-report
require_shell_success("final run comparison")

## 13. Display the main CSV and selected figures

In [ ]:
import pandas as pd
from IPython.display import display, Image as DisplayImage
comparison_dir = sorted((REPO_DIR / "runs" / "comparisons").glob("*_final"))[-1]
display(pd.read_csv(comparison_dir / "main_comparison.csv"))
for figure in [comparison_dir / "method_success_comparison.png", comparison_dir / "transfer_gap.png"]:
    if figure.is_file():
        display(DisplayImage(filename=str(figure)))

## 14. Build the main paper-style report

In [ ]:
!latexmk -pdf -interaction=nonstopmode -halt-on-error -outdir=reports reports/main.tex
require_shell_success("main report build")

## 15. Build the detailed surrogate report

In [ ]:
!latexmk -pdf -interaction=nonstopmode -halt-on-error -outdir=reports reports/surrogate_notes.tex
require_shell_success("surrogate report build")

## 16. Sync runs, reports, PDFs, plots, and videos to Drive

In [ ]:
for experiment in ["mappo_nominal", "mappo_uniform_dr", "mappo_failure_dr"]:
    print(copy_completed_run(latest_run(experiment)))
sync_all()

## 17. Finished
Confirm the Drive copies and compiled PDFs, record failed or excluded runs in `surrogate_notes.tex`, and disconnect the Colab runtime.